# ==================================================
# Task 1: Decision Tree Baseline
# ==================================================

##Imports & Load Data

In [2]:
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load cleaned dataset
df = pd.read_csv("/content/cleaned_data.csv")

print("First 5 Rows:")
display(df.head())

print("\nShape:", df.shape)

First 5 Rows:


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832
1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232
2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000
3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360
4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080



Shape: (1274, 11)


## Create Features & Targets

In [3]:
# Feature Matrix
X = df.drop(columns=["Price"])

# Regression Target (used later if needed)
y_reg = df["Price"]

# Classification Target
y_clf = (df["Price"] > df["Price"].median()).astype(int)

print("Feature Matrix Shape:", X.shape)
print("Classification Target Shape:", y_clf.shape)

print("\nClass Distribution:")
display(y_clf.value_counts())

Feature Matrix Shape: (1274, 10)
Classification Target Shape: (1274,)

Class Distribution:


,count
Price,
0,644
1,630


##One-Hot Encoding

In [4]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("Encoded Shape:", X.shape)

Encoded Shape: (1274, 338)


##Train/Test Split & Scaling

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Shape:", X_train_scaled.shape)
print("Testing Shape:", X_test_scaled.shape)

Training Shape: (1019, 338)
Testing Shape: (255, 338)


##Decision Tree Baseline Model

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create model
dt_model = DecisionTreeClassifier(random_state=42)

# Train
dt_model.fit(X_train_scaled, y_train)

# Predictions
train_pred = dt_model.predict(X_train_scaled)
test_pred = dt_model.predict(X_test_scaled)

# Accuracy
train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Training Accuracy:", round(train_accuracy, 4))
print("Testing Accuracy :", round(test_accuracy, 4))

Training Accuracy: 0.998
Testing Accuracy : 0.8863


# ==================================================
# Task 2: Controlled Decision Tree
# ==================================================

## Create controlled Decision Tree Model

In [7]:
controlled_dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

# Train
controlled_dt.fit(X_train_scaled, y_train)

# Predictions
train_pred_controlled = controlled_dt.predict(X_train_scaled)
test_pred_controlled = controlled_dt.predict(X_test_scaled)

# Accuracy
train_accuracy_controlled = accuracy_score(
    y_train,
    train_pred_controlled
)

test_accuracy_controlled = accuracy_score(
    y_test,
    test_pred_controlled
)

print("Training Accuracy:", round(train_accuracy_controlled, 4))
print("Testing Accuracy :", round(test_accuracy_controlled, 4))

Training Accuracy: 0.8813
Testing Accuracy : 0.898


# ==================================================
# Task 3: Gini vs Entropy
# ==================================================

In [8]:
# Gini Decision Tree
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)

gini_pred = gini_tree.predict(X_test_scaled)

gini_accuracy = accuracy_score(y_test, gini_pred)

# Entropy Decision Tree
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

entropy_tree.fit(X_train_scaled, y_train)

entropy_pred = entropy_tree.predict(X_test_scaled)

entropy_accuracy = accuracy_score(y_test, entropy_pred)

print("Gini Test Accuracy   :", round(gini_accuracy, 4))
print("Entropy Test Accuracy:", round(entropy_accuracy, 4))

Gini Test Accuracy   : 0.8941
Entropy Test Accuracy: 0.898


# ==================================================
# Task 4: Random Forest
# ==================================================

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Create Random FOrest Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# Train
rf_model.fit(X_train_scaled, y_train)

# Predictions
train_pred_rf = rf_model.predict(X_train_scaled)
test_pred_rf = rf_model.predict(X_test_scaled)

# Probabilities
test_prob_rf = rf_model.predict_proba(X_test_scaled)[: ,1]

# Accuracy
train_accuracy_rf = accuracy_score(y_train, train_pred_rf)
test_accuracy_rf = accuracy_score(y_test, test_pred_rf)

# ROC-AUC
roc_auc_rf = roc_auc_score(y_test, test_prob_rf)

print("Training Accuracy:", round(train_accuracy_rf, 4))
print("Testing Accuracy :", round(test_accuracy_rf, 4))
print("ROC-AUC          :", round(roc_auc_rf, 4))

Training Accuracy: 0.9176
Testing Accuracy : 0.9098
ROC-AUC          : 0.9717


##Feature Importance

In [10]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

top5 = importance.sort_values(
    by="Importance",
    ascending=False
).head(5)

print("Top 5 Important Features:")
display(top5)

Top 5 Important Features:


,Feature,Importance
1,Ram,0.159243
23,TypeName_Notebook,0.129593
2,Weight,0.074105
191,Memory_1TB HDD,0.062813
21,TypeName_Gaming,0.047444


## ==================================================
## Task 4a: Gradient Boosting
## ==================================================

In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Create Model
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Train
gb_model.fit(X_train_scaled, y_train)

# Predictions
train_pred_gb = gb_model.predict(X_train_scaled)
test_pred_gb = gb_model.predict(X_test_scaled)

# Probabilities
test_prob_gb = gb_model.predict_proba(X_test_scaled)[: ,1]

# Metrics
train_accuracy_gb = accuracy_score(y_train, train_pred_gb)
test_accuracy_gb = accuracy_score(y_test, test_pred_gb)
roc_auc_gb = roc_auc_score(y_test, test_prob_gb)

print("Training Accuracy:", round(train_accuracy_gb, 4))
print("Testing Accuracy :", round(test_accuracy_gb, 4))
print("ROC-AUC          :", round(roc_auc_gb, 4))

Training Accuracy: 0.9382
Testing Accuracy : 0.9255
ROC-AUC          : 0.9791


## ==================================================
## Task 4b: Feature Ablation Study
## ==================================================

##Find the Bottom 5 Features

In [12]:
# Bottom 5 least important features
bottom5 = importance.sort_values(
    by="Importance"
).head(5)

print("Bottom 5 Least Important Features:")
display(bottom5)

Bottom 5 Least Important Features:


,Feature,Importance
304,Gpu_Nvidia GeForce GTX 940M,0.0
308,Gpu_Nvidia GeForce GTX 960<U+039C>,0.0
312,Gpu_Nvidia GeForce GTX 980,0.0
41,ScreenResolution_IPS Panel Full HD 1920x1200,0.0
291,Gpu_Nvidia GeForce 940M,0.0


## Remove Them

In [13]:
# Feature names to remove
remove_features = bottom5["Feature"].tolist()

print("Features Removed:")
display(remove_features)

Features Removed:


['Gpu_Nvidia GeForce GTX 940M',
 'Gpu_Nvidia GeForce GTX 960<U+039C>',
 'Gpu_Nvidia GeForce GTX 980 ',
 'ScreenResolution_IPS Panel Full HD 1920x1200',
 'Gpu_Nvidia GeForce 940M']

##Remove from Train & Test

In [14]:
X_train_reduced = X_train.drop(columns=remove_features)
X_test_reduced = X_test.drop(columns=remove_features)

## Scale Again

In [15]:
scaler_reduced = StandardScaler()

X_train_reduced_scaled = scaler_reduced.fit_transform(X_train_reduced)
X_test_reduced_scaled = scaler_reduced.transform(X_test_reduced)

## Train Again

In [16]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(
    X_train_reduced_scaled,
    y_train
)

prob_reduced = rf_reduced.predict_proba(
    X_test_reduced_scaled
)[:,1]

auc_reduced = roc_auc_score(
    y_test,
    prob_reduced
)

print("Original Random Forest AUC :", round(roc_auc_rf,4))
print("Reduced Random Forest AUC  :", round(auc_reduced,4))

Original Random Forest AUC : 0.9717
Reduced Random Forest AUC  : 0.9693


# ==================================================
# Task 5: Cross-Validated Model Comparison
# ==================================================

In [17]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

results = []

for name, model in models.items():

  scores = cross_val_score(
      model,
      X_train_scaled,
      y_train,
      cv=cv,
      scoring="roc_auc"
  )

  results.append(
      {
          "Model": name,
          "Mean AUC": scores.mean(),
          "Std AUC": scores.std()
      }
  )

cv_results = pd.DataFrame(results)

print("5-Fold Cross Validation Results:")
display(cv_results.round(4))

5-Fold Cross Validation Results:


,Model,Mean AUC,Std AUC
0,Logistic Regression,0.9478,0.0093
1,Decision Tree,0.8914,0.0266
2,Random Forest,0.9578,0.0044
3,Gradient Boosting,0.9559,0.0101


# ==================================================
# Task 6: GridSearchCV + Pipeline
# ==================================================

##Imports

In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

##Parameter Grid

In [19]:
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

##Build Pipeline

In [20]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

##Grid Search

In [21]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

##Fit

In [22]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('simpleimputer',
                                        SimpleImputer(strategy='median')),
                                       ('standardscaler', StandardScaler()),
                                       ('randomforestclassifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'randomforestclassifier__max_depth': [5, 10, None],
                         'randomforestclassifier__min_samples_leaf': [1, 5],
                         'randomforestclassifier__n_estimators': [50, 100,
                                                                  200]},
             scoring='roc_auc')

##Print Results

In [ ]:
Print Results

In [23]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest ROC-AUC:")
print(round(grid_search.best_score_, 4))

Best Parameters:
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 100}

Best ROC-AUC:
0.9632


# ==================================================
# Task 7: Manual Learning Curve
# ==================================================

In [24]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_results = []

best_pipeline = grid_search.best_estimator_

for frac in fractions:

    n_samples = int(frac * len(X_train))

    X_subset = X_train.iloc[:n_samples]
    y_subset = y_train.iloc[:n_samples]

    # Train
    best_pipeline.fit(X_subset, y_subset)

    # Training probabilities
    train_prob = best_pipeline.predict_proba(X_subset)[:, 1]

    # Testing probabilities
    test_prob = best_pipeline.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(
        y_subset,
        train_prob
    )

    test_auc = roc_auc_score(
        y_test,
        test_prob
    )

    learning_results.append({
        "Training Fraction": frac,
        "Training AUC": train_auc,
        "Test AUC": test_auc
    })

learning_df = pd.DataFrame(learning_results)

print("Manual Learning Curve:")
display(learning_df.round(4))

Manual Learning Curve:


,Training Fraction,Training AUC,Test AUC
0,0.2,1.0,0.9545
1,0.4,1.0,0.9676
2,0.6,1.0,0.9776
3,0.8,1.0,0.9789
4,1.0,1.0,0.9804


# ==================================================
# Task 8: Model Serialization
# ==================================================

##Save the Best Model

In [25]:
import joblib

# Save model
joblib.dump(best_pipeline, "best_model.pkl")

print("Model saved successfully.")

Model saved successfully.


##Load the Model

In [26]:
# Load model
loaded_model = joblib.load("best_model.pkl")

print("Model loaded successfully.")

Model loaded successfully.


##Predict on Two Samples

In [27]:
# Select two sample rows
sample_data = X.iloc[[0, 1]]

# Predict
predictions = loaded_model.predict(sample_data)

print("Predictions:")
print(predictions)

Predictions:
[1 0]
